# Greyscale ablation — clinical-concordance decider (harmonised vs original)

**Isolated experiment, branch `greyscale-experiment`.** NEW analysis only; no Phase 1–4 module,
config, prediction, or report is modified. All outputs under `greyscale_experiment/`. CPU fine.

Design C1 softened the zero-shot collapse (harmonised OLIVES grade-0 ≈ 53% vs the original 96%),
at ~0.02 in-domain QWK cost. **This decides whether that spread is genuine transferable DR
severity signal or a new brightness-driven artifact** — mirroring Phase 4b
(`src/analysis/dr_failure_characterisation.py`) on the harmonised predictions and comparing
original vs harmonised on every test. The verdict rests on the **sign** of grade↔CST concordance
(genuine severity ⇒ positive ρ; the original was ρ ≈ −0.492, wrong sign) and the
**clinical-vs-brightness head-to-head** — not on the softened distribution alone.

In [ ]:
# Setup: mount Drive, restore the repo on the greyscale-experiment branch, cd in.
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR = '/content/dr-dissertation'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin && git checkout greyscale-experiment && git pull

## Run STEP 1 introspection + the full concordance analysis
The module prints STEP 1 (harmonised filename, columns, row/valid counts, grade distribution;
confirms the original predictions + Phase 4b JSON + OLIVES tensor are readable), then runs the
Phase 4b-parallel tests on the harmonised predictions and the original, writing
`greyscale_concordance.json`, `greyscale_concordance_report.md`, and 4 figures.

In [ ]:
!python -m src.analysis.greyscale_concordance --config configs/greyscale_eval.yaml

## Display the report + figures

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path

root = Path('/content/drive/MyDrive/dissertation/greyscale_experiment')
display(Markdown((root / 'greyscale_concordance_report.md').read_text()))

figs = [
    'greyscale_concordance_fig1_grade_vs_cst.png',
    'greyscale_concordance_fig2_grade_vs_burden.png',
    'greyscale_concordance_fig3_grade_vs_brightness.png',
    'greyscale_concordance_fig4_rho_summary.png',
]
for name in figs:
    p = root / 'figures' / name
    if p.exists():
        display(Image(str(p)))

## Done
Verdict (recovered-signal / new-artifact / partial) is at the top of the report, with the key
numbers: harmonised grade↔CST ρ and its **sign**, grade↔brightness ρ, the clinical-vs-brightness
head-to-head, and the original-vs-harmonised table. A softened distribution alone is **not**
validated grading — on a modality with no ground-truth grade, a positive is strong indirect
evidence, not proof. Commit/push from PowerShell on `greyscale-experiment`.